# 🦎 Galápagos Classifier — MobileNetV3Small

Lee imágenes desde el **banco compartido** (`galapagos_wildlife/raw_images/`).
Para descargar o agregar imágenes usa `00_download_dataset.ipynb`.

**Fases:**
1. Setup (lee banco, prepara dataset train/val)
2. Entrenamiento MobileNetV3Small
3. Export TFLite float16

**Salida:** `MyDrive/galapagos_wildlife/classifier/models/`


## Celda 0 — Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/galapagos_wildlife'
CLASSIFIER_DIR = f'{DRIVE_ROOT}/classifier'
os.makedirs(f'{CLASSIFIER_DIR}/models', exist_ok=True)
print(f'Drive OK: {DRIVE_ROOT}')


## Celda 1 — Setup e Imports

In [ ]:
import importlib, subprocess, sys
def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
ensure('Pillow', 'PIL')
ensure('tqdm')

import os, json, time, shutil, random
import numpy as np
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

IMAGES_DIR    = Path(f'{DRIVE_ROOT}/raw_images')
DATA_DIR      = Path('/content/data')
MODELS_DIR    = Path(f'{CLASSIFIER_DIR}/models')
IMG_SIZE      = 224
BATCH_SIZE    = 32
MIN_IMGS      = 5      # mínimo para incluir una clase (bajo para permitir subespecies raras)
random.seed(42)

# Leer SPECIES desde species_config.json (fuente única de verdad)
config_path = Path(f'{DRIVE_ROOT}/species_config.json')
if not config_path.exists():
    raise FileNotFoundError(
        'species_config.json no encontrado.\n'
        'Ejecuta 00_download_dataset.ipynb primero.'
    )
config = json.loads(config_path.read_text())

# Mapear directorio del banco → species config
# Formato directorio: "38_Chelonoidis_niger_porteri" → split en primer _ → "Chelonoidis niger porteri"
available_dirs = {}
if IMAGES_DIR.exists():
    for d in IMAGES_DIR.iterdir():
        if d.is_dir():
            name_part = d.name.split('_', 1)[-1].replace('_', ' ')
            available_dirs[name_part.lower()] = d

valid_species = []
print(f'{"ID":>4}  {"Especie":<44} {"Imgs":>6}  {"Estado"}')
print('-' * 70)
for sp in config['species']:
    sid  = sp['id']
    sci, en, es = sp['scientific'], sp['en'], sp['es']
    src_dir = available_dirs.get(sci.lower())
    if src_dir and src_dir.exists():
        n = len(list(src_dir.glob('*.jpg')))
        if n >= MIN_IMGS:
            status = f'OK ({n})'
            valid_species.append((sci, en, es, src_dir, n))
        else:
            status = f'pocas ({n}) — skip' if n > 0 else 'sin imgs — skip'
    else:
        n, status = 0, 'sin dir — skip'
    print(f'{sid:>4}  {en:<44} {n:>6}   {status}')

print(f'\nEspecies incluidas: {len(valid_species)}/{len(config["species"])}')
total_imgs = sum(n for *_, n in valid_species)
print(f'Total imágenes:     {total_imgs}')

if len(valid_species) < 5:
    print('AVISO: Pocas especies. Ejecuta 00_download_dataset.ipynb primero.')

## Celda 2 — Preparar Dataset (train/val desde banco)

In [ ]:
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR   = DATA_DIR / 'val'

print('Copiando imágenes del banco a /content/data ...')
class_sizes = {}  # cls_key → n_train (para class_weight)

for sci_name, en_name, es_name, src_dir, total_n in valid_species:
    cls_key = sci_name.replace(' ', '_')
    imgs = list(src_dir.glob('*.jpg'))
    random.shuffle(imgs)
    split = max(1, int(len(imgs) * 0.8))
    for split_name, split_imgs in [('train', imgs[:split]), ('val', imgs[split:])]:
        dst = DATA_DIR / split_name / cls_key
        dst.mkdir(parents=True, exist_ok=True)
        for img in split_imgs:
            target = dst / img.name
            if not target.exists():
                shutil.copy(img, target)
    n_train = len(list((DATA_DIR / 'train' / cls_key).glob('*.jpg')))
    n_val   = len(list((DATA_DIR / 'val'   / cls_key).glob('*.jpg')))
    class_sizes[cls_key] = n_train
    flag = ' ⚠ pocas' if n_train < 20 else ''
    print(f'  {en_name:<40} train:{n_train:4d}  val:{n_val:3d}{flag}')

train_ds = keras.utils.image_dataset_from_directory(
    str(TRAIN_DIR), image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='int', shuffle=True, seed=42,
)
val_ds = keras.utils.image_dataset_from_directory(
    str(VAL_DIR), image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='int', shuffle=False,
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(f'\nClases: {NUM_CLASSES}  |  train: {len(train_ds)} batches  |  val: {len(val_ds)} batches')

# ── Pesos de clase para desbalance ─────────────────────────────────────────
# Con porteri (1500+) vs hoodensis (6) hay desbalance extremo.
# class_weight compensa: clases raras tienen mayor peso en el loss.
counts = np.array([class_sizes.get(cn, 1) for cn in CLASS_NAMES], dtype=np.float32)
total  = counts.sum()
# Peso inversamente proporcional a frecuencia, normalizado
weights = (total / (NUM_CLASSES * counts))
class_weight = {i: float(w) for i, w in enumerate(weights)}
print(f'\nClass weights (muestra):')
for i, cn in enumerate(CLASS_NAMES):
    if weights[i] > 2.0 or i < 3:
        print(f'  [{i:02d}] {cn:<50} n={counts[i]:5.0f}  w={weights[i]:.2f}')
print(f'  ... ({NUM_CLASSES} clases total)')

# ── Augmentation y normalización ─────────────────────────────────────────
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='augmentation')

normalize  = layers.Rescaling(scale=1.0/127.5, offset=-1.0)
AUTOTUNE   = tf.data.AUTOTUNE
train_ds   = train_ds.map(
    lambda x, y: (normalize(augment(x, training=True)), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)
val_ds = val_ds.map(
    lambda x, y: (normalize(x), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)
print('Dataset listo')

## Celda 3 — Modelo MobileNetV3Small

In [ ]:
base_model = keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=False,  # normalizamos nosotros
)
base_model.trainable = False  # fase 1: solo entrenar cabeza

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_1')
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

model = keras.Model(inputs, outputs, name='galapagos_classifier')
model.summary(expand_nested=False)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

## Celda 4 — Entrenamiento (fase 1 + fine-tune)

In [ ]:
# ─── Fase 1: entrenar cabeza (base congelada) ──────────────────────
print('=== FASE 1: entrenar cabeza (base congelada) ===')
hist1 = model.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    class_weight=class_weight,   # compensa el desbalance entre especies
    callbacks=[
        keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            '/content/best_model.keras', save_best_only=True,
            monitor='val_accuracy', verbose=0
        ),
    ]
)
best_acc = max(hist1.history['val_accuracy'])
print(f'Fase 1 completada — mejor val_accuracy: {best_acc*100:.1f}%')

# ─── Fase 2: fine-tune (últimas capas del backbone) ────────────────
print('\n=== FASE 2: fine-tune (últimas 40 capas del backbone) ===')
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

hist2 = model.fit(
    train_ds,
    epochs=30,                   # más épocas con más clases
    validation_data=val_ds,
    class_weight=class_weight,   # mantener compensación en fine-tune
    callbacks=[
        keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(
            '/content/best_model.keras', save_best_only=True,
            monitor='val_accuracy', verbose=1
        ),
    ]
)

best_acc2 = max(hist2.history['val_accuracy'])
print(f'Fase 2 completada — mejor val_accuracy: {best_acc2*100:.1f}%')

# Guardar mejor modelo en Drive
if os.path.exists('/content/best_model.keras'):
    shutil.copy('/content/best_model.keras', f'{CLASSIFIER_DIR}/models/best_model.keras')
    print(f'Modelo guardado en Drive: {CLASSIFIER_DIR}/models/')

## Celda 5 — Evaluación

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f'Val accuracy: {acc*100:.1f}%  —  Val loss: {loss:.4f}')

# Matriz de confusión (top errores)
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true, y_pred = [], []
for x_batch, y_batch in val_ds:
    preds = model.predict(x_batch, verbose=0)
    y_true.extend(y_batch.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Nombres cortos para el gráfico
short_names = [cn.split('_')[1] if '_' in cn else cn for cn in CLASS_NAMES]

print('\n=== Top clases con más errores ===')
cm = confusion_matrix(y_true, y_pred)
np.fill_diagonal(cm, 0)
error_counts = cm.sum(axis=1)
top_errors = np.argsort(error_counts)[::-1][:10]
for idx in top_errors:
    if error_counts[idx] > 0:
        print(f'  {CLASS_NAMES[idx]:45s} {error_counts[idx]} errores')

## Celda 6 — Export TFLite float16

In [ ]:
best_model = keras.models.load_model('/content/best_model.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

MODEL_PATH  = '/content/galapagos_classifier.tflite'
LABELS_PATH = '/content/labels.txt'

with open(MODEL_PATH, 'wb') as f:
    f.write(tflite_model)

# Labels: scientific_name|common_en|common_es  (una línea por clase, en orden CLASS_NAMES)
# CLASS_NAMES está ordenado alfabéticamente por keras (por nombre de directorio)
name_map = {sci.replace(' ', '_'): (sci, en, es) for sci, en, es, _, _n in valid_species}
with open(LABELS_PATH, 'w') as f:
    for cls_name in CLASS_NAMES:
        if cls_name in name_map:
            sci, en, es = name_map[cls_name]
        else:
            sci = en = es = cls_name.replace('_', ' ')
        f.write(f'{sci}|{en}|{es}\n')

size_mb = os.path.getsize(MODEL_PATH) / 1024 / 1024
print(f'TFLite: {size_mb:.1f} MB  |  {NUM_CLASSES} clases')
print(f'\nClases exportadas:')
with open(LABELS_PATH) as f:
    for i, line in enumerate(f):
        sci, en, es = line.strip().split('|')
        print(f'  [{i:02d}] {en}')

# Guardar en Drive
shutil.copy(MODEL_PATH,  f'{CLASSIFIER_DIR}/models/galapagos_classifier.tflite')
shutil.copy(LABELS_PATH, f'{CLASSIFIER_DIR}/models/labels.txt')

# Log
log_entry = {
    'timestamp': __import__('datetime').datetime.now().isoformat(),
    'num_classes': NUM_CLASSES,
    'size_mb': round(size_mb, 2),
    'class_names': CLASS_NAMES,
    'val_accuracy': round(best_acc2 * 100, 1),
}
log_path = Path(f'{CLASSIFIER_DIR}/training_log.json')
history = json.loads(log_path.read_text()) if log_path.exists() else []
history.append(log_entry)
log_path.write_text(json.dumps(history, indent=2))
print(f'\nGuardado en Drive: {CLASSIFIER_DIR}/models/')

## Celda 7 — Descargar para Flutter

In [ ]:
from google.colab import files
print('Descargando archivos para Flutter...')
files.download(LABELS_PATH)
files.download(MODEL_PATH)
print('\nColoca en assets/ml/ del proyecto Flutter:')
print('  galapagos_classifier.tflite -> assets/ml/galapagos_classifier.tflite')
print('  labels.txt                  -> assets/ml/labels.txt')


## Celda 8 — (Opcional) Validación manual

In [ ]:
import io

def predict_tflite(image_path: str, top_k: int = 5):
    """Prueba el modelo TFLite con una imagen local."""
    interp = tf.lite.Interpreter(model_path=MODEL_PATH)
    interp.allocate_tensors()
    in_d  = interp.get_input_details()[0]
    out_d = interp.get_output_details()[0]

    img = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32)
    arr = (arr / 127.5) - 1.0  # normalizar igual que en entrenamiento
    arr = np.expand_dims(arr, 0)
    if in_d['dtype'] == np.float16:
        arr = arr.astype(np.float16)

    interp.set_tensor(in_d['index'], arr)
    interp.invoke()
    probs = interp.get_tensor(out_d['index'])[0]

    top_indices = np.argsort(probs)[::-1][:top_k]
    with open(LABELS_PATH) as f:
        labels = [l.strip().split('|') for l in f.readlines()]

    print(f'Predicciones para: {image_path}')
    for idx in top_indices:
        sci, en, es = labels[idx] if idx < len(labels) else (str(idx), str(idx), str(idx))
        print(f'  {probs[idx]*100:5.1f}%  {en} ({sci})')

# Ejemplo: descomentar y cambiar el path
# predict_tflite('/content/data/raw/Sula_nebouxii/0001.jpg')